# Project: Gaussian process regression 🌈
A Gaussian process puts a probability distribution over **functions**. Condition it on a few noisy observations and you get a prediction *with honest error bars*, wide where you have no data. Implement the RBF kernel and the posterior equations with a Cholesky decomposition, then watch the uncertainty and the effect of the length-scale.

Topic: [[Gaussian Processes]], [[Kernel Methods]], [[Bayesian Inference]] · guide note: [[Project - Gaussian Process Regression]]

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing
rng = np.random.default_rng(0)
f = lambda x: np.sin(x) + 0.3 * x
X = np.sort(rng.choice(np.r_[np.linspace(-5, -1, 50), np.linspace(2, 5, 50)], 12, replace=False))[:, None]   # a gap in the middle
y = f(X[:, 0]) + rng.normal(0, 0.2, len(X)); Xs = np.linspace(-7, 7, 300)[:, None]

## 1. The RBF kernel\n$k(x, x') = \sigma_f^2\exp\!\big(-\frac{\lVert x-x'\rVert^2}{2\ell^2}\big)$ for all pairs of rows of `A` and `B`.

In [ ]:
def rbf(A, B, length=1.0, sigma_f=1.0):
    # TODO 1: pairwise squared distances → RBF kernel matrix
    raise NotImplementedError  # TODO 1

## 2. The posterior\nWith $K = k(X,X) + \sigma_n^2I$, $K_* = k(X, X_*)$, $K_{**} = k(X_*,X_*)$: mean $= K_*^\top K^{-1}y$, covariance $= K_{**} - K_*^\top K^{-1}K_*$. Never invert: use the Cholesky factor $K = LL^\top$ and triangular solves. Return `(mean, cov)`.

In [ ]:
def gp_posterior(X, y, Xs, noise=0.2, **kern):
    # TODO 2: Cholesky-based GP posterior mean and covariance
    raise NotImplementedError  # TODO 2

In [ ]:
def _t_gp():
    K = rbf(np.array([[0.0], [1.0]]), np.array([[0.0], [2.0]]), length=1.0, sigma_f=2.0)
    m, cov = gp_posterior(X, y, Xs, noise=0.2, length=1.0, sigma_f=1.0)
    sk = GaussianProcessRegressor(C(1.0, "fixed") * RBF(1.0, "fixed"), alpha=0.04, optimizer=None).fit(X, y)
    ms, ss = sk.predict(Xs, return_std=True)
    return np.allclose(K, [[4, 4 * np.exp(-2)], [4 * np.exp(-0.5), 4 * np.exp(-0.5)]]) and np.allclose(m, ms, atol=1e-6) and np.allclose(np.sqrt(np.clip(np.diag(cov), 0, None)), ss, atol=1e-5)
check("GP posterior", _t_gp, "Must match scikit-learn's GaussianProcessRegressor with the same fixed kernel and noise.")
if ready("GP posterior"):
    fig, ax = plt.subplots(1, 3, figsize=(16, 3.6), sharey=True)
    for a, ell in zip(ax, (0.3, 1.0, 3.0)):
        m, cov = gp_posterior(X, y, Xs, length=ell); sd = np.sqrt(np.clip(np.diag(cov), 0, None))
        a.fill_between(Xs[:, 0], m - 2 * sd, m + 2 * sd, alpha=.3); a.plot(Xs, m); a.plot(Xs, f(Xs), "k--", lw=1); a.scatter(X, y, c="r", zorder=3)
        a.set_title(f"length-scale ℓ = {ell}")
    plt.show()
    m, cov = gp_posterior(X, y, Xs, length=1.0); sd = np.sqrt(np.clip(np.diag(cov), 0, None))
    gap, near = sd[np.abs(Xs[:, 0] - 0.5) < 0.3].mean(), sd[np.abs(Xs[:, 0] + 3) < 0.3].mean()
    print(f"posterior std in the data gap (x≈0.5): {gap:.2f}   near data (x≈−3): {near:.2f}")
    check("honest uncertainty", lambda: gap > 3 * near, "The error bars should be much wider in the gap than near the data.")

<details><summary>▶ Solution</summary>

```python
def rbf(A, B, length=1.0, sigma_f=1.0):
    d2 = ((A[:, None, :] - B[None, :, :]) ** 2).sum(-1)
    return sigma_f ** 2 * np.exp(-d2 / (2 * length ** 2))
```

```python
def gp_posterior(X, y, Xs, noise=0.2, **kern):
    K = rbf(X, X, **kern) + noise ** 2 * np.eye(len(X))
    Ks = rbf(X, Xs, **kern); Kss = rbf(Xs, Xs, **kern)
    L = np.linalg.cholesky(K)
    alpha = np.linalg.solve(L.T, np.linalg.solve(L, y))
    mean = Ks.T @ alpha
    v = np.linalg.solve(L, Ks)
    cov = Kss - v.T @ v
    return mean, cov
```
</details>

## Stretch goals
- Maximise the log marginal likelihood over ℓ and σ_f (grid search) to choose the kernel automatically.
- Draw 5 sample functions from the posterior (`np.random.multivariate_normal(mean, cov)`).
- Fit a GP to the Aalborg temperature data from [[Project - Forecasting Aalborg Temperature]] with a periodic kernel.